## Statistical data analysis. Exam 

**(!) <span style="color:blue"> Add your name to the file name instead of [Name]</span>**</br>
Example: Stat_analysis_24-25_exam_Ivanov Ivan.ipynb

Submit your notebook to Smart LMS https://edu.hse.ru/course/view.php?id=230834

Unlocked web sites to visit:</br>
* google search
* https://docs.scipy.org/doc/scipy/tutorial/index.html
* https://docs.python.org/3/library/
* https://numpy.org/
* https://pandas.pydata.org/docs/user_guide/index.html
* https://matplotlib.org/
* https://plotly.com/python/

In [ ]:
import numpy as np
import random
import plotly.graph_objects as go

**Question 1 [14 pts]**. Implement classes defined in Problem 1.

In [ ]:
class basic_discrete():
    def __init__(self):
        pass
    
    def prediction(self, x):
        pass

Random-walk related classes

In [ ]:
class random_walk(basic_discrete):
    def __init__(self, init_val, prob):
        super().__init__()
        self.R0 = init_val
        self.p = prob
    
    def prediction(self, x):
        return self.R0 + np.sum(x)
    
    def exp_transform(self, x):
        Rn = self.prediction(x)
        return np.exp(Rn)
    
class symmetric_rw(random_walk):
    def __init__(self, init_val):
        super().__init__(init_val, 0.5)

Create an object of *random_walk* class with $R_0=0, p=0.8$ and output member variable $p$.

In [ ]:
rw = random_walk(0, 0.8)
rw.p

Create an object of *symmetric_rw* class with  $R_0=2$, output probability  $p$, *prediction* and *exp_tranform* on a given list $x$ below.

In [ ]:
x = [-1, 2, -2]

rw_s = symmetric_rw(2)
print(rw_s.p, rw_s.prediction(x), rw_s.exp_transform(x))

Binomial model related classes

In [ ]:
class binomial(basic_discrete):
    def __init__(self, init_val, up, down, rate, prob):
        super().__init__()
        self.S0 = init_val
        self.u = up
        self.d = down
        self.r = rate
        self.q = prob
    
    def prediction(self, x):
        return self.S0*np.prod(x)
    
    def log_transform(self, x):
        Sn = self.prediction(x)
        return np.log(Sn)
    
class binomial_risk_neutral(binomial):
    def __init__(self, init_val, up, down, rate):
        rn_prob = (1+rate-down)/(up-down)
        super().__init__(init_val, up, down, rate, rn_prob)

Create an object of *binomial* class with $S_0=1, u = 1.1, d= 0.9, r = 0.05, q = 0.4$ and output member variable $q$.

In [ ]:
binom_ = binomial(1, 1.1, 0.9, 0.05, 0.4)
binom_.q

Create an object of *binomial_risk_neutral* class with  $S_0=2,u = 1.1, d= 0.9, r = 0.05$, output member variable $q$, *prediction* and *log_tranform* on a given list $y$ below.

In [ ]:
y = [1, 2, 2.5]
binom_rn = binomial_risk_neutral(2, 1.1, 0.9, 0.05)
print(binom_rn.q, binom_rn.prediction(y), binom_rn.log_transform(y))

**Question 2 [25 pts]** Consider financial instrument with payoff $f_n=\overline{S}-K$ defined in Problem 3 under Binomial model. Class for this instrument is in place, but you need to complete definition of payoff.

In [ ]:
class instrument:
    def __init__ (self, K, T):
        self.strike = K
        self.maturity = T
        
    def payoff(self, S):
        return np.mean(S)-self.strike

Define an instrument with parameters given below:

In [ ]:
K = 105
T = 12
instr = instrument(K, T)

Write *analytic_pricer* which derives from given class *pricer*. No need to write *model* class, use given model parameters $S_0,\,u,\,d,\,r$ in pv() function as they are.

In [ ]:
class pricer:
    def __init__(self):
        pass
        
    def pv(self, instr):
        pass

In [ ]:
S0 = 100
u = 1.1
d = 0.9
r = 0.05

In [ ]:
class analytic_pricer(pricer):
    def __init__(self):
        pass
    
    def pv(self, instr):
        K = instr.strike
        n = instr.maturity
        
        expected_payoff = S0*((1+r)**(n+1)-1)/(r*(n+1)) - K
        return expected_payoff/((1+r)**n)

**Output**: analytic PV of an instrument.

In [ ]:
an_pricer = analytic_pricer()
analytic_pv = an_pricer.pv(instr)
analytic_pv

Write *monte_carlo_pricer* which derives from class *pricer*. No need to write *model* class, use given model parameters $S_0,\,u,\,d,\,r$ in pv() function as they are.<br/>
 *binom_simulation* function below is a helper.

In [ ]:
def binom_simulation(n_paths, max_time, S0, u, d, q):
    paths = []
    for i in range(n_paths):
        ys = [S0] + random.choices([d, u], weights = [1-q, q], k = max_time)
        path = np.cumprod(ys)
        paths.append(path)
    
    return np.array(paths)

In [ ]:
class monte_carlo_pricer(pricer):
    def __init__(self, N_paths):
        self.N = N_paths  
        
    def pv(self, instr):
        n = instr.maturity
        
        q = (1+r-d)/(u-d)
        s_paths = binom_simulation(self.N, n, S0, u, d, q)
        
        payoffs = [instr.payoff(path) for path in s_paths]
        discounted_payoffs = np.array(payoffs)/((1+r)**n)
        return np.mean(discounted_payoffs)        

**Output**: Monte-Carlo PV of an instrument and absolute difference between analytic and Monte-Carlo price.

In [ ]:
n_paths = 100000
mc_pricer = monte_carlo_pricer(n_paths)
mc_pv = mc_pricer.pv(instr)
print(mc_pv, abs(mc_pv-analytic_pv))

**Question 3 [11 pts]**

Consider differential equation and finite difference scheme from Problem 4.</br>
Implement finite difference scheme. Output 4 scatterplots (mode=lines) on the same plot: exact solution $y=y(x)$ and 3 solutions $(y_0,\ldots, y_{2n})$ of finite difference schemes for $n=10,\,30$ and $80$.

In [ ]:
initial = 0.5
T = 2
ns = [10, 30, 80]

solutions = []
for n in ns:
    n_pts = n*T + 1
    x_ = np.linspace(0, T, n_pts)
    h = 1/n
    yh = [initial]
    for i in range(1, n_pts+1):
        yh.append(yh[i-1]/(1-2*h*(i*h-1)))
    solutions.append([x_, yh])

In [ ]:
plot = go.Figure()

x_ = np.linspace(0,T,1001)
y_ = 0.5*np.exp(x_**2 - 2*x_)
graph = go.Scatter(x = x_, y = y_, name = 'exact')
plot.add_trace(graph)

for i in range(len(ns)):
    graph1 = go.Scatter(x = solutions[i][0], y = solutions[i][1], mode ='lines', name = f'N={ns[i]}')
    plot.add_trace(graph1)

plot.show()